# Collinearity

This notebook assesses the collinearity among the variables used in the vulnerability indicator. It loads all raster layers representing the sensitivity and lack of adaptive capacity variables and samples pixel values from each raster while linking them to country identifiers. The sampled pixel values are then aggregated to country-level averages to create a dataset suitable for evaluating relationships among the variables.

Using this dataset, the notebook calculates pairwise Spearman rank correlations and variance inflation factors (VIF) to assess potential multicollinearity among the indicators. The analysis is conducted at both the country level and the pixel level (based on sampled pixels) to evaluate whether correlations arise primarily from spatial patterns or from aggregation effects. It then creates:

- table showing the Spearman correlation matrix among all variables at the country level

- table showing the variance inflation factors (VIF) for all variables at the country level

- table showing the Spearman correlation matrix among all variables at the pixel level (based on sampled pixels)

- table showing the variance inflation factors (VIF) for all variables at the pixel level

## How to run
1. Put the required input files in the same folder as this notebook (or edit the paths in the **Configuration** cell below).
2. Run the cells from top to bottom.

## Required files
- `country_id.tif`
- `bii_5000m.tif`
- `wdpa_5000m.tif`
- `landmark_5000m.tif`
- `kba_5000m.tif`
- `poverty_5000m.tif`
- `water_risk_5000m.tif`
- `conflict_5000m.tif`
- `edi_5000m.tif` 
- `landrights_5000m.tif`
- `rule_of_law_5000m.tif`
- `World_Countries_(Generalized)_8414823838130214587.gpkg` (or your country layer)

In [ ]:
# Configuration (edit these paths if needed)

COUNTRY_ID = 'vulnerability_indicator\\countries\\country_id.tif'  

raster_paths=['sensitivity\\biodiversity_intactness\\bii_5000m.tif',
    'sensitivity\\protected_areas\\wdpa_5000m.tif',
    'sensitivity\\kba\\kba_5000m.tif',
    'sensitivity\\water_risk\\water_risk_5000m.tif',
    'sensitivity\\poverty\\poverty_5000m.tif',
    'sensitivity\\ind_com_lands\\landmark_5000m.tif' ,
    'lackof_adapt\\landrights\\landrights_5000m.tif',
    'lackof_adapt\\environmental_democracy\\edi_5000m.tif',
    'lackof_adapt\\rule_of_law\\rule_of_law_5000m.tif',
    'lackof_adapt\\conflict\\conflict_5000m.tif'
    ]

names = ["bii", "wdpa", "landmark", "kba", "poverty", "water_risk",
         "conflict", "edi", "landrights", "rule_of_law"]

#other parameters
dst_nodata = -9999.0
window_size = 1024
target_samples = 150_000

In [ ]:
#import packages
import os, math
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

In [ ]:
# calculate collinearity of variables
country_id_path = COUNTRY_ID

def sample_rasters_to_df_by_country_id(
    raster_paths, names, country_id_path,
    target_samples=100_000, window_size=1024, nodata=-9999.0, seed=42
):

    rng = np.random.default_rng(seed)  # random generator for reproducible sampling

    # open all variable rasters and the country ID raster
    srcs = [rasterio.open(p) for p in raster_paths]
    cid_src = rasterio.open(country_id_path)

    ref = srcs[0]  # reference raster for size/grid

    collected = []       # store sampled blocks
    collected_n = 0      # count sampled pixels

    try:
        # number of raster windows (chunks) to iterate over
        n_rows = math.ceil(ref.height / window_size)
        n_cols = math.ceil(ref.width / window_size)

        for r in range(n_rows):
            for c in range(n_cols):

                # define raster window
                x_off = c * window_size
                y_off = r * window_size
                w = min(window_size, ref.width - x_off)
                h = min(window_size, ref.height - y_off)
                window = Window(x_off, y_off, w, h)

                # read country IDs
                cid = cid_src.read(1, window=window).astype(np.int32)
                inside = cid > 0  # keep pixels belonging to countries

                if not np.any(inside):
                    continue

                # read all variable rasters for this window
                arrays = [s.read(1, window=window) for s in srcs]

                # build validity mask (inside country + no nodata)
                valid = inside.copy()
                for a in arrays:
                    valid &= np.isfinite(a) & (a != nodata)

                # indices of valid pixels
                idx = np.flatnonzero(valid)
                if idx.size == 0:
                    continue

                # stop if enough samples collected
                remaining = target_samples - collected_n
                if remaining <= 0:
                    break

                # randomly sample pixels from this window
                take = min(idx.size, max(500, remaining // 10))
                chosen = rng.choice(idx, size=take, replace=False)

                # extract sampled country IDs and raster values
                cols = [cid.reshape(-1)[chosen].astype(np.int32)]
                for a in arrays:
                    cols.append(a.reshape(-1)[chosen].astype(np.float32))

                # stack into array and store
                block = np.column_stack(cols)
                collected.append(block)
                collected_n += block.shape[0]

            # stop outer loop if sample size reached
            if collected_n >= target_samples:
                break

    finally:
        # close all raster files
        for s in srcs:
            s.close()
        cid_src.close()

    # ensure some samples were collected
    if not collected:
        raise RuntimeError("No valid samples found (check nodata, masks, alignment).")

    # combine sampled blocks into one array
    X = np.vstack(collected)

    # trim if slightly above target
    if X.shape[0] > target_samples:
        X = X[:target_samples, :]

    # convert to DataFrame
    df = pd.DataFrame(X, columns=["country_id"] + names)

    return df


# run sampling and create pixel table
df = sample_rasters_to_df_by_country_id(
    raster_paths, names, country_id_path,
    target_samples=target_samples,
    window_size=window_size,
    nodata=dst_nodata
)

In [ ]:
# Aggregate sampled pixels to country-level averages
country_df = df.groupby("country_id")[names].mean(numeric_only=True)

# Count how many pixels were sampled per country
counts = df.groupby("country_id").size()

# Keep only countries with enough sampled pixels (>=50)
keep = counts[counts >= 50].index

# Recompute country means using only those countries
country_df = df[df["country_id"].isin(keep)].groupby("country_id")[names].mean()

In [ ]:
def correlation_color(v):
    a = abs(v)

    # strength → intensity
    if a < 0.10:
        intensity = 0
    elif a < 0.30:
        intensity = 1
    elif a < 0.50:
        intensity = 2
    elif a < 0.70:
        intensity = 3
    elif a < 0.90:
        intensity = 4
    else:
        intensity = 5

    # blue = negative, red = positive
    blues = ["#EFF6FF", "#DBEAFE", "#93C5FD", "#60A5FA", "#2563EB", "#1E3A8A"]
    reds  = ["#FEF2F2", "#FEE2E2", "#FCA5A5", "#F87171", "#DC2626", "#B91C1C"]

    return blues[intensity] if v < 0 else reds[intensity]

In [ ]:
# pretty names for variables
name = {
    "wdpa": "PA",
    "conflict": "Conflict",
    "edi": "EDI",
    "landrights": "Land rights",
    "rule_of_law": "Rule of law",
    "bii": "BII",
    "kba": "KBA",
    "landmark": "IPLC lands",
    "poverty": "Poverty",
    "water_risk": "Water risk",
}
# rename rows/columns for nicer display
def apply_names(obj, map):
    if isinstance(obj, pd.DataFrame):
        out = obj.copy()
        out.index = [map.get(str(x), str(x)) for x in out.index]
        out.columns = [map.get(str(x), str(x)) for x in out.columns]
        return out
    elif isinstance(obj, pd.Series):
        out = obj.copy()
        out.index = [map.get(str(x), str(x)) for x in out.index]
        return out
    return obj

# custom color map for correlation table
def modern_diverging_cmap():
    colors = ["#5B8FF9", "#F5F5F5", "#F4664A"]
    return LinearSegmentedColormap.from_list("modern_div", colors, N=256)

# plot styled correlation matrix table
def plot_correlation_table(corr, outpath=None):
    corr = apply_names(corr, name).round(3)
    vals = corr.values
    n = vals.shape[0]

    fig, ax = plt.subplots(figsize=(1.25 * n + 1.5, 1.1 * n + 1.0), dpi=200)
    ax.axis("off")


    table = ax.table(
        cellText=corr.values,
        rowLabels=corr.index,
        colLabels=corr.columns,
        cellLoc="center",
        loc="center",
    )

    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.15, 1.35)

    # style header, row labels, and data cells
    for (r, c), cell in table.get_celld().items():
        if r == 0:
            cell.set_facecolor("#F3F4F6")
            cell.set_text_props(fontweight="semibold", color="#111827")
            cell.set_edgecolor("#E5E7EB")
            cell.set_linewidth(0.8)
            continue

        if c == -1:
            cell.set_facecolor("#F9FAFB")
            cell.set_text_props(fontweight="semibold", color="#111827")
            cell.set_edgecolor("#E5E7EB")
            cell.set_linewidth(0.8)
            continue

        v = float(vals[r - 1, c])
        cell.set_facecolor(correlation_color(v))
        cell.set_edgecolor("#E5E7EB")
        cell.set_linewidth(0.6)

        if abs(v) >= 0.7:
            cell.set_text_props(fontweight="semibold", color="#111827")
        else:
            cell.set_text_props(color="#111827")

    ax.set_title("Spearman correlation matrix, country level", fontsize=14, fontweight="semibold", pad=12, color="#111827")

    if outpath:
        fig.savefig(outpath, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)

# compute variance inflation factors
def compute_vif(country_df):
    X = (country_df - country_df.mean()) / country_df.std(ddof=0)  # standardize
    X = X.replace([np.inf, -np.inf], np.nan).dropna()              # drop invalid rows

    X_np = X.to_numpy()
    vifs = [(col, variance_inflation_factor(X_np, i)) for i, col in enumerate(X.columns)]
    vif_df = pd.DataFrame(vifs, columns=["Variable", "VIF"]).sort_values("VIF", ascending=False)

    vif_df["Variable"] = vif_df["Variable"].astype(str).map(lambda x: name.get(x, x))
    return vif_df

# plot styled VIF table
def plot_vif_table(vif_df, outpath=None):
    vf = vif_df.copy()
    vf["VIF"] = vf["VIF"].astype(float).round(2)

    fig_h = 0.9 + 0.45 * len(vf)
    fig, ax = plt.subplots(figsize=(7.0, fig_h), dpi=200)
    ax.axis("off")

    table = ax.table(
        cellText=vf.values,
        colLabels=vf.columns,
        cellLoc="center",
        loc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1.1, 1.35)

    # style header
    for j in range(len(vf.columns)):
        cell = table[(0, j)]
        cell.set_facecolor("#F3F4F6")
        cell.set_text_props(fontweight="semibold", color="#111827")
        cell.set_edgecolor("#E5E7EB")
        cell.set_linewidth(0.8)

    # style rows and highlight high VIF
    for i in range(1, len(vf) + 1):
        for j in range(len(vf.columns)):
            cell = table[(i, j)]
            cell.set_edgecolor("#E5E7EB")
            cell.set_linewidth(0.6)
            cell.set_text_props(color="#111827")
            cell.set_facecolor("white")

        vif_val = float(vf.iloc[i - 1, 1])
        vif_cell = table[(i, 1)]
        if vif_val >= 10:
            vif_cell.set_facecolor("#FEE2E2")
        elif vif_val >= 5:
            vif_cell.set_facecolor("#FEF3C7")

    ax.set_title("Variance Inflation Factors (VIF), Country Level", fontsize=14, fontweight="semibold", pad=12, color="#111827")

    if outpath:
        fig.savefig(outpath, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)

# calculate Spearman correlation matrix
corr = country_df.corr(method="spearman")
plot_correlation_table(corr, outpath="correlation_table_spearman_country.png")

# calculate VIF table
vif_df = compute_vif(country_df)
plot_vif_table(vif_df, outpath="vif_table_country.png")

In [ ]:
# pixel-level collinearity tables

# pretty names for variables
name = {
    "wdpa": "PA",
    "conflict": "Conflict",
    "edi": "EDI",
    "landrights": "Land rights",
    "rule_of_law": "Rule of law",
    "bii": "BII",
    "kba": "KBA",
    "landmark": "IPLC lands",
    "poverty": "Poverty",
    "water_risk": "Water risk",
}

# rename rows/columns for nicer display
def apply_names(obj, map):
    if isinstance(obj, pd.DataFrame):
        out = obj.copy()
        out.index = [map.get(str(x), str(x)) for x in out.index]
        out.columns = [map.get(str(x), str(x)) for x in out.columns]
        return out
    elif isinstance(obj, pd.Series):
        out = obj.copy()
        out.index = [map.get(str(x), str(x)) for x in out.index]
        return out
    return obj

# custom color map for correlation table
def modern_diverging_cmap():
    colors = ["#5B8FF9", "#F5F5F5", "#F4664A"]
    return LinearSegmentedColormap.from_list("modern_div", colors, N=256)

# plot styled correlation matrix table
def plot_correlation_table(corr, outpath=None):
    corr = apply_names(corr, name).round(3)
    vals = corr.values
    n = vals.shape[0]

    fig, ax = plt.subplots(figsize=(1.25 * n + 1.5, 1.1 * n + 1.0), dpi=200)
    ax.axis("off")

    cmap = modern_diverging_cmap()
    norm = plt.Normalize(vmin=-1, vmax=1)

    table = ax.table(
        cellText=corr.values,
        rowLabels=corr.index,
        colLabels=corr.columns,
        cellLoc="center",
        loc="center",
    )

    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.15, 1.35)

    # style header, row labels, and data cells
    for (r, c), cell in table.get_celld().items():
        if r == 0:
            cell.set_facecolor("#F3F4F6")
            cell.set_text_props(fontweight="semibold", color="#111827")
            cell.set_edgecolor("#E5E7EB")
            cell.set_linewidth(0.8)
            continue

        if c == -1:
            cell.set_facecolor("#F9FAFB")
            cell.set_text_props(fontweight="semibold", color="#111827")
            cell.set_edgecolor("#E5E7EB")
            cell.set_linewidth(0.8)
            continue

        v = float(vals[r - 1, c])
        cell.set_facecolor(correlation_color(v))
        cell.set_edgecolor("#E5E7EB")
        cell.set_linewidth(0.6)

        if abs(v) >= 0.7:
            cell.set_text_props(fontweight="semibold", color="#111827")
        else:
            cell.set_text_props(color="#111827")

    ax.set_title("Spearman correlation matrix, pixel l level", fontsize=14, fontweight="semibold", pad=12, color="#111827")

    if outpath:
        fig.savefig(outpath, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)

# compute variance inflation factors
def compute_vif(pixel_df):
    X = (pixel_df - pixel_df.mean()) / pixel_df.std(ddof=0)  # standardize
    X = X.replace([np.inf, -np.inf], np.nan).dropna()        # drop invalid rows

    X_np = X.to_numpy()
    vifs = [(col, variance_inflation_factor(X_np, i)) for i, col in enumerate(X.columns)]
    vif_df = pd.DataFrame(vifs, columns=["Variable", "VIF"]).sort_values("VIF", ascending=False)

    vif_df["Variable"] = vif_df["Variable"].astype(str).map(lambda x: name.get(x, x))
    return vif_df

# plot styled VIF table
def plot_vif_table(vif_df, outpath=None):
    vf = vif_df.copy()
    vf["VIF"] = vf["VIF"].astype(float).round(2)

    fig_h = 0.9 + 0.45 * len(vf)
    fig, ax = plt.subplots(figsize=(7.0, fig_h), dpi=200)
    ax.axis("off")

    table = ax.table(
        cellText=vf.values,
        colLabels=vf.columns,
        cellLoc="center",
        loc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1.1, 1.35)

    # style header
    for j in range(len(vf.columns)):
        cell = table[(0, j)]
        cell.set_facecolor("#F3F4F6")
        cell.set_text_props(fontweight="semibold", color="#111827")
        cell.set_edgecolor("#E5E7EB")
        cell.set_linewidth(0.8)

    # style rows and highlight high VIF
    for i in range(1, len(vf) + 1):
        for j in range(len(vf.columns)):
            cell = table[(i, j)]
            cell.set_edgecolor("#E5E7EB")
            cell.set_linewidth(0.6)
            cell.set_text_props(color="#111827")
            cell.set_facecolor("white")

        vif_val = float(vf.iloc[i - 1, 1])
        vif_cell = table[(i, 1)]
        if vif_val >= 10:
            vif_cell.set_facecolor("#FEE2E2")
        elif vif_val >= 5:
            vif_cell.set_facecolor("#FEF3C7")

    ax.set_title("Variance inflation factors (VIF), pixel level", fontsize=14, fontweight="semibold", pad=12, color="#111827")

    if outpath:
        fig.savefig(outpath, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)

# keep only pixel-level variable columns
pixel_df = df[names].copy()

# remove invalid rows
pixel_df = pixel_df.replace([np.inf, -np.inf], np.nan).dropna()

# calculate Spearman correlation matrix
corr = pixel_df.corr(method="spearman")
plot_correlation_table(corr, outpath="correlation_table_spearman_pixel.png")

# calculate VIF table
vif_df = compute_vif(pixel_df)
plot_vif_table(vif_df, outpath="vif_table_pixel.png")